# 🧠 Laboratorio 1.2 – Building a Boolean Search Engine on a Real Collection

**Corso:** Information Retrieval – Laurea Magistrale in Informatica  
**Università:** Roma “Tor Vergata”  
**Docente:** Danilo Croce

---

## 🎯 Obiettivo del laboratorio

In questo laboratorio costruiamo una prima versione di un motore di ricerca booleano su una collezione reale di documenti testuali.

Dopo aver introdotto nel laboratorio precedente i concetti fondamentali dell’Information Retrieval su una collezione ridotta e controllata, passiamo ora a un contesto più realistico: useremo un sottoinsieme del dataset **20 Newsgroups** per affrontare alcuni problemi concreti di indicizzazione e ricerca.

L’obiettivo è mostrare come passare da una collezione di documenti grezzi a una struttura di indicizzazione interrogabile tramite query booleane.

In particolare, nel laboratorio vedremo come:

- acquisire una collezione di documenti da un dataset reale
- applicare una pipeline di **preprocessing**
- costruire un **indice inverso non posizionale**
- salvare e ricaricare la struttura dati dell’indice
- eseguire query booleane con operatori come **AND**, **OR** e **NOT**
- discutere limiti e possibili miglioramenti dell’implementazione

---

## 📚 Dataset: 20 Newsgroups

Per questo laboratorio useremo un sottoinsieme del dataset **20 Newsgroups**, una collezione classica molto usata negli esperimenti di text mining e Information Retrieval.

Il dataset contiene circa 20.000 documenti distribuiti su 20 gruppi tematici.  
Per semplicità, inizieremo lavorando su una sola categoria, così da concentrarci meglio sui passaggi di preprocessing, indicizzazione e query processing.

---

## 🛠️ Struttura del laboratorio

Nel notebook costruiremo progressivamente una pipeline composta da questi passi:

1. caricamento dei documenti  
2. ispezione della collezione  
3. preprocessing del testo  
4. costruzione dell’indice inverso unigram  
5. ispezione di termini e postings lists  
6. esecuzione di query booleane
7. analisi dei limiti dell’implementazione  
8. esercizi finali

---

In [ ]:
import re
import json
from collections import defaultdict

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nltk.download('punkt_tab')

from sklearn.datasets import fetch_20newsgroups

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Caricamento del dataset

Useremo una sola categoria del dataset per mantenere il laboratorio leggero e facilmente esplorabile.

In questo modo possiamo concentrarci sugli aspetti fondamentali della pipeline di Information Retrieval senza introdurre troppa complessità fin dall’inizio.

In [ ]:
category = ["comp.graphics"]

dataset = fetch_20newsgroups(
    subset="train",
    categories=category,
    remove=()   # non rimuoviamo nulla all'inizio: vogliamo vedere i documenti grezzi
)

documents = dataset.data

print("Selected categories:", category)
print("Number of documents:", len(documents))

Selected categories: ['comp.graphics']
Number of documents: 584


In [ ]:
print("First raw document:\n")
print(documents[0][:3000])

First raw document:

From: bbs.mirage@tsoft.net (Jerry Lee)
Subject: Cobra 2.0 1-b-1 Video card HELP ME!!!!
Organization: The TSoft BBS and Public Access Unix, +1 415 969 8238
Lines: 22

Does ANYONE out there in Net-land have any information on the Cobra 2.20 
card?  The sticker on the end of the card reads
        Model: Cobra 1-B-1
        Bios:  Cobra v2.20

I Havn't been able to find anything about it from anyone!  If you have 
any information on how to get a hold of the company which produces the 
card or know where any drivers are for it, PLEASE let me know!

As far as I can tell, it's a CGA card that is taking up 2 of my 16-bit 
ISA slots but when I enable the test patterns, it displays much more than 
the usualy 4 CGA colors... At least 16 from what I can count.. Thanks!

              .------------------------------------------.
              : Internet: jele@eis.calstate.edu          :
              :           bbs.mirage@gilligan.tsoft.net  :
              :           bbs.mi

## Osservazione

I documenti del dataset non contengono solo testo “pulito”, ma anche molte informazioni accessorie, ad esempio:

- header
- indirizzi email
- metadata
- punteggiatura
- numeri
- parole molto frequenti e poco informative

Questa osservazione motiva la necessità di una fase di **preprocessing** prima della costruzione dell’indice.

## Preprocessing

Prima di costruire l’indice, dobbiamo trasformare i documenti grezzi in una forma più adatta alla ricerca.

In questo laboratorio useremo una pipeline di preprocessing composta da alcuni passaggi classici:

- rimozione dell’header
- conversione in minuscolo
- normalizzazione dei numeri
- rimozione della punteggiatura
- tokenizzazione
- rimozione delle stopwords
- rimozione di token troppo corti
- stemming

L’obiettivo non è ottenere la pipeline “perfetta”, ma mostrare come queste trasformazioni influenzano la rappresentazione finale dei documenti e quindi anche l’indice inverso.

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

## Rimozione dell'header

Molti documenti del dataset contengono una parte iniziale fatta di metadati, come:

- `From:`
- `Subject:`
- `Organization:`
- `Lines:`

Queste informazioni possono essere utili in altri contesti, ma qui vogliamo concentrarci soprattutto sul contenuto testuale del documento.

Per questo definiamo una funzione che prova a rimuovere tutto ciò che precede la prima riga vuota.

In [ ]:
def remove_header(text):
    parts = text.split("\n\n", 1)
    if len(parts) == 2:
        return parts[1]
    return text

## Funzioni elementari di preprocessing

Definiamo ora alcune trasformazioni di base che applicheremo ai documenti.

In [ ]:
def convert_lower_case(text):
    return text.lower()


def convert_numbers(text):
    number_map = {
        "0": " zero ",
        "1": " one ",
        "2": " two ",
        "3": " three ",
        "4": " four ",
        "5": " five ",
        "6": " six ",
        "7": " seven ",
        "8": " eight ",
        "9": " nine ",
    }
    for digit, word in number_map.items():
        text = text.replace(digit, word)
    return text


def remove_punctuation(text):
    return re.sub(r"[^\w\s]", " ", text)


def remove_extra_spaces(text):
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
def tokenize(text):
    return word_tokenize(text)


def remove_stop_words(tokens):
    return [token for token in tokens if token not in stop_words]


def remove_single_characters(tokens):
    return [token for token in tokens if len(token) > 1]


def apply_stemming(tokens):
    return [stemmer.stem(token) for token in tokens]

## Pipeline completa di preprocessing

Mettiamo insieme tutti i passaggi in una singola funzione.

Useremo il parametro `is_query` per distinguere due casi:

- **documento**: possiamo rimuovere anche l’header
- **query**: non dobbiamo rimuovere nessun header, perché la query è già una stringa molto breve

La funzione restituirà una lista di token preprocessati.

In [ ]:
def preprocess(text, is_query=False):
    if not is_query:
        text = remove_header(text)

    text = convert_lower_case(text)
    text = convert_numbers(text)
    text = remove_punctuation(text)
    text = remove_extra_spaces(text)

    tokens = tokenize(text)
    tokens = remove_stop_words(tokens)
    tokens = remove_single_characters(tokens)
    tokens = apply_stemming(tokens)

    return tokens

## Esempio di preprocessing su un documento reale

Vediamo ora che effetto ha la pipeline su uno dei documenti del dataset.

Questo passaggio è utile perché mostra concretamente quanto il testo venga trasformato prima della costruzione dell’indice.

In [ ]:
raw_example = documents[0]
processed_example = preprocess(raw_example, is_query=False)

print("First 1200 raw characters:\n")
print(raw_example[:1200])

print("\n" + "=" * 80 + "\n")

print("First 80 preprocessed tokens:\n")
print(processed_example[:80])

First 1200 raw characters:

From: bbs.mirage@tsoft.net (Jerry Lee)
Subject: Cobra 2.0 1-b-1 Video card HELP ME!!!!
Organization: The TSoft BBS and Public Access Unix, +1 415 969 8238
Lines: 22

Does ANYONE out there in Net-land have any information on the Cobra 2.20 
card?  The sticker on the end of the card reads
        Model: Cobra 1-B-1
        Bios:  Cobra v2.20

I Havn't been able to find anything about it from anyone!  If you have 
any information on how to get a hold of the company which produces the 
card or know where any drivers are for it, PLEASE let me know!

As far as I can tell, it's a CGA card that is taking up 2 of my 16-bit 
ISA slots but when I enable the test patterns, it displays much more than 
the usualy 4 CGA colors... At least 16 from what I can count.. Thanks!

              .------------------------------------------.
              : Internet: jele@eis.calstate.edu          :
              :           bbs.mirage@gilligan.tsoft.net  :
              :          

## Osservazione

Dopo il preprocessing:

- molti metadati scompaiono
- le maiuscole vengono uniformate
- la punteggiatura viene rimossa
- le stopwords più comuni vengono eliminate
- parole con la stessa radice tendono a collassare tramite stemming

Questo rende la rappresentazione più compatta e più adatta alla costruzione di un indice inverso.

Naturalmente, ogni scelta di preprocessing ha anche effetti collaterali: in alcuni casi si perde informazione utile.  
Per questo, nei sistemi reali, la progettazione della pipeline di preprocessing è una decisione importante.

In [ ]:
query_example = "Place or authority and welcome"
processed_query_example = preprocess(query_example, is_query=True)

print("Original query:")
print(query_example)

print("\nPreprocessed query tokens:")
print(processed_query_example)

Original query:
Place or authority and welcome

Preprocessed query tokens:
['place', 'author', 'welcom']


## Applicazione del preprocessing all'intera collezione

Ora applichiamo la pipeline a tutti i documenti selezionati.

Per comodità, memorizziamo il risultato in un dizionario che associa a ogni `docID` la lista dei token preprocessati.

In [ ]:
tokenized_documents = {}

for doc_id, text in enumerate(documents):
    tokenized_documents[doc_id] = preprocess(text, is_query=False)

print("Number of preprocessed documents:", len(tokenized_documents))

Number of preprocessed documents: 584


In [ ]:
for doc_id in range(3):
    print(f"Doc {doc_id} -> first 40 tokens:")
    print(tokenized_documents[doc_id][:40])
    print()

Doc 0 -> first 40 tokens:
['anyon', 'net', 'land', 'inform', 'cobra', 'two', 'two', 'zero', 'card', 'sticker', 'end', 'card', 'read', 'model', 'cobra', 'one', 'one', 'bio', 'cobra', 'two', 'two', 'zero', 'havn', 'abl', 'find', 'anyth', 'anyon', 'inform', 'get', 'hold', 'compani', 'produc', 'card', 'know', 'driver', 'pleas', 'let', 'know', 'far', 'tell']

Doc 1 -> first 40 tokens:
['hi', 'everyon', 'one', 'touch', 'problem', 'post', 'last', 'week', 'guess', 'question', 'clear', 'like', 'describ', 'detail', 'offset', 'ellips', 'locu', 'center', 'circl', 'roll', 'ellips', 'word', 'distanc', 'ellips', 'offset', 'everywher', 'problem', 'come', 'geometr', 'measur', 'probe', 'use', 'tip', 'probe', 'ball', 'comput', 'output', 'posit', 'ball', 'center']

Doc 2 -> first 40 tokens:
['hi', 'short', 'look', 'fast', 'assembl', 'code', 'line', 'circl', 'draw', 'svga', 'graphic', 'complet', 'think', 'simpl', 'fast', 'molecular', 'graphic', 'program', 'write', 'pc', 'clone', 'ball', 'stick', 'type', 'r

## Mini esercizio

Osserva i primi documenti preprocessati e prova a rispondere:

1. quali informazioni vengono eliminate dal preprocessing?
2. quali parole risultano oggetto di stemming?
3. ci sono casi in cui il preprocessing potrebbe rimuovere informazione utile?

Queste domande sono importanti perché il preprocessing non è mai neutrale: cambia il modo in cui i documenti verranno rappresentati e quindi anche il comportamento del motore di ricerca.

## Costruzione dell'indice inverso unigram

Una volta preprocessati i documenti, possiamo costruire un **indice inverso non posizionale**.

### Idea
Per ogni termine vogliamo memorizzare la lista dei documenti in cui compare.

Questa struttura è detta **inverted index** e costituisce la base del retrieval booleano.

Nel caso di un **indice unigram**, ogni chiave del dizionario corrisponde a un singolo termine preprocessato.

In [ ]:
postings = defaultdict(list)

for doc_id, tokens in tokenized_documents.items():
    unique_tokens = set(tokens)  # evitiamo duplicati dello stesso docID nella stessa posting list. Ma è ridondante?
    for token in unique_tokens:
        postings[token].append(doc_id)

# Ordiniamo le posting lists per docID. Ma non sono già ordinati "by construction"?
for token in postings:
    postings[token] = sorted(postings[token])

print("Index built.")
print("Vocabulary size:", len(postings))

Index built.
Vocabulary size: 8603


## Osservazione

Nel codice precedente usiamo `set(tokens)` prima di aggiornare l'indice.

### Perché?
Perché in un indice inverso non posizionale ogni documento deve comparire **al massimo una volta** nella posting list di un termine.

Ad esempio, se il termine `graphic` compare 10 volte nello stesso documento, nella posting list vogliamo comunque inserire solo il relativo `docID` una sola volta.

In [ ]:
print("Some example postings lists:\n")

example_terms = ["graphic", "imag", "file", "format", "window"]

for term in example_terms:
    print(f"{term:12s} -> {postings.get(term, [])[:30]}")

Some example postings lists:

graphic      -> [2, 3, 6, 8, 11, 14, 17, 19, 26, 28, 34, 37, 42, 43, 48, 51, 69, 72, 74, 76, 77, 79, 83, 87, 88, 93, 101, 103, 104, 109]
imag         -> [0, 3, 4, 7, 9, 13, 17, 18, 22, 25, 26, 34, 36, 44, 45, 47, 49, 52, 53, 54, 55, 61, 64, 76, 78, 79, 85, 86, 89, 92]
file         -> [3, 4, 8, 13, 15, 25, 26, 29, 35, 39, 44, 46, 52, 57, 58, 59, 60, 61, 65, 67, 70, 71, 74, 75, 76, 78, 79, 81, 83, 86]
format       -> [6, 15, 26, 35, 43, 61, 76, 78, 87, 89, 93, 108, 116, 120, 136, 138, 140, 145, 154, 161, 166, 180, 202, 210, 213, 214, 223, 242, 255, 257]
window       -> [3, 7, 26, 32, 35, 52, 60, 69, 70, 76, 99, 105, 106, 119, 120, 128, 134, 135, 144, 154, 155, 158, 163, 166, 188, 210, 212, 213, 218, 229]


## Dictionary e postings lists

L'indice inverso può essere visto come composto da due elementi logici:

- un **dictionary**, che contiene i termini
- un insieme di **postings lists**, una per ciascun termine

Nel nostro caso rappresentiamo tutto con un singolo dizionario Python, ma concettualmente è utile distinguere queste due componenti.

In [ ]:
document_frequency = {term: len(doc_ids) for term, doc_ids in postings.items()}

print("Some document frequencies:\n")

for term in example_terms:
    print(f"{term:12s} -> df = {document_frequency.get(term, 0)}")

Some document frequencies:

graphic      -> df = 190
imag         -> df = 139
file         -> df = 158
format       -> df = 71
window       -> df = 74


## Document frequency

La **document frequency** di un termine è il numero di documenti in cui quel termine compare.

Questa informazione è utile perché:

- ci dice quanto il termine è diffuso nella collezione
- permette di identificare termini più o meno selettivi
- sarà utile per strategie di ottimizzazione delle query

In [ ]:
sorted_terms_by_df = sorted(document_frequency.items(), key=lambda x: x[1])

print("10 least frequent terms:\n")
for term, df in sorted_terms_by_df[:10]:
    print(f"{term:15s} -> df = {df}")

print("\n10 most frequent terms:\n")
for term, df in sorted_terms_by_df[-10:]:
    print(f"{term:15s} -> df = {df}")

10 least frequent terms:

gilligan        -> df = 1
ei              -> df = 1
sticker         -> df = 1
jele            -> df = 1
calstat         -> df = 1
slot            -> df = 1
tsoft           -> df = 1
cga             -> df = 1
cobra           -> df = 1
thetech         -> df = 1

10 most frequent terms:

edu             -> df = 239
nine            -> df = 240
eight           -> df = 249
five            -> df = 277
six             -> df = 279
four            -> df = 300
zero            -> df = 330
three           -> df = 362
two             -> df = 366
one             -> df = 402


## Ispezione dell'indice

L'ispezione manuale di alcune postings lists è utile per capire:

- se il preprocessing ha prodotto termini plausibili
- se l'indice contiene i documenti attesi
- quali termini sono molto frequenti e quali sono rari

Questa fase è importante anche in un contesto reale, perché consente di verificare che la pipeline stia funzionando correttamente.

In [ ]:
def get_posting(word, postings_index):
    return postings_index.get(word, [])

def print_word_postings(word, postings_index):
    processed_tokens = preprocess(word, is_query=True)

    if len(processed_tokens) == 0:
        print("The input disappears after preprocessing.")
        return

    if len(processed_tokens) > 1:
        print("Warning: multiple tokens produced:", processed_tokens)

    processed_word = processed_tokens[0]
    posting_list = get_posting(processed_word, postings_index)

    print("Original input:", word)
    print("Processed term:", processed_word)
    print("Document Frequency:", len(posting_list))
    print("Postings List:", posting_list)

In [ ]:
print_word_postings("welcome", postings)

Original input: welcome
Processed term: welcom
Document Frequency: 10
Postings List: [11, 13, 105, 185, 340, 366, 445, 496, 499, 575]


## Perché il termine cambia?

Nel caso precedente, la parola immessa dall'utente può cambiare dopo il preprocessing.

Per esempio:

- `welcome` può diventare `welcom`
- `authority` può diventare `author`

Questo accade per effetto dello **stemming**, che riduce parole morfologicamente simili a una radice comune.

Dal punto di vista del retrieval, questo può essere utile perché aumenta le possibilità di match tra query e documenti.

In [ ]:
test_terms = ["welcome", "authority", "graphics", "images", "files"]

for word in test_terms:
    print("=" * 60)
    print_word_postings(word, postings)
    print()

Original input: welcome
Processed term: welcom
Document Frequency: 10
Postings List: [11, 13, 105, 185, 340, 366, 445, 496, 499, 575]

Original input: authority
Processed term: author
Document Frequency: 26
Postings List: [11, 34, 44, 61, 69, 71, 76, 103, 142, 166, 167, 169, 175, 198, 209, 213, 234, 295, 351, 366, 381, 425, 502, 517, 550, 575]

Original input: graphics
Processed term: graphic
Document Frequency: 190
Postings List: [2, 3, 6, 8, 11, 14, 17, 19, 26, 28, 34, 37, 42, 43, 48, 51, 69, 72, 74, 76, 77, 79, 83, 87, 88, 93, 101, 103, 104, 109, 118, 119, 120, 121, 124, 125, 130, 132, 134, 138, 142, 146, 148, 151, 154, 156, 163, 164, 166, 169, 174, 182, 184, 188, 192, 194, 195, 199, 204, 205, 209, 212, 213, 215, 220, 224, 226, 231, 232, 234, 245, 247, 250, 258, 259, 263, 264, 271, 272, 275, 276, 277, 278, 284, 287, 288, 292, 302, 303, 305, 307, 312, 313, 314, 315, 316, 319, 322, 325, 327, 329, 334, 340, 341, 343, 350, 353, 356, 357, 358, 362, 365, 366, 368, 370, 371, 373, 374, 376,

## Salvataggio dell'indice

Una volta costruito l'indice, può essere utile salvarlo su file.

Questo passaggio permette di:
- evitare di ricostruire l'indice ogni volta
- riusare la struttura in notebook successivi
- separare la fase di indexing dalla fase di querying

In [ ]:
output_filename = "20newsgroups_comp_graphics_unigram_postings.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(dict(postings), f)

print(f"Index saved to: {output_filename}")

Index saved to: 20newsgroups_comp_graphics_unigram_postings.json


In [ ]:
with open(output_filename, "r", encoding="utf-8") as f:
    loaded_postings = json.load(f)

print("Loaded index.")
print("Vocabulary size:", len(loaded_postings))

Loaded index.
Vocabulary size: 8603


## Nota pratica

Dopo il caricamento da JSON, le chiavi restano termini testuali e le posting lists vengono ricostruite come liste Python.

Questo è sufficiente per i nostri esperimenti.

In sistemi reali, tuttavia, le strutture dati per gli indici sono spesso molto più compatte e ottimizzate, anche dal punto di vista della memoria e dell'accesso su disco.

In [ ]:
sample_term = "graphic"

print("Original postings:", postings.get(sample_term, [])[:20])
print("Loaded postings:  ", loaded_postings.get(sample_term, [])[:20])

Original postings: [2, 3, 6, 8, 11, 14, 17, 19, 26, 28, 34, 37, 42, 43, 48, 51, 69, 72, 74, 76]
Loaded postings:   [2, 3, 6, 8, 11, 14, 17, 19, 26, 28, 34, 37, 42, 43, 48, 51, 69, 72, 74, 76]


## Mini esercizio

Prova a ispezionare manualmente alcuni termini del dominio `comp.graphics`, ad esempio:

- `render`
- `imag`
- `color`
- `polygon`
- `file`

Per ciascun termine osserva:
- la forma preprocessata
- la document frequency
- la lunghezza della posting list

### Domanda
Quali termini ti sembrano più selettivi?  
Quali invece sembrano troppo frequenti per essere particolarmente utili in una query booleana?

## Query processing

Ora che abbiamo costruito l’indice inverso, possiamo usarlo per eseguire query booleane.

Nel laboratorio precedente abbiamo visto il caso ideale, in cui le postings lists sono ordinate e si possono intersecare con un algoritmo di **merge**.

Qui vogliamo invece costruire una prima versione di un motore di ricerca booleano su una collezione reale, mantenendo l’implementazione semplice e facilmente leggibile.

### Obiettivo
Supportare query del tipo:

- `place`
- `place and authority`
- `place or authority`
- `place and not authority`
- `place or authority and welcome`

Questa implementazione sarà volutamente semplice e ci permetterà, più avanti, di discutere i suoi limiti.

## Recupero della posting list di un termine

Quando l’utente scrive un termine in input, non possiamo cercarlo direttamente nell’indice senza prima applicare lo stesso preprocessing usato per i documenti.

Questo è importante perché nell’indice i termini sono memorizzati nella loro forma preprocessata, ad esempio dopo:
- lowercase
- rimozione di stopwords
- stemming

In [ ]:
print_word_postings("welcome", postings)

Original input: welcome
Processed term: welcom
Document Frequency: 10
Postings List: [11, 13, 105, 185, 340, 366, 445, 496, 499, 575]


## Implementazione degli operatori booleani

Definiamo ora tre operazioni di base:

- **AND**: intersezione
- **OR**: unione
- **NOT**: complemento rispetto all’insieme di tutti i documenti

Per semplicità, in questa prima implementazione useremo operazioni basate su `set`.

Questa scelta è comoda e leggibile, ma non è la più efficiente possibile dal punto di vista algoritmico.

In [ ]:
# In questa prima implementazione usiamo un set con tutti i docID della collezione.
# Ci serve soprattutto per calcolare in modo semplice il complemento di una posting list,
# cioè l'operatore NOT.
# In una versione più vicina ai sistemi classici di IR, lavoreremmo invece in modo più
# diretto su posting lists ordinate, evitando quando possibile conversioni a set.
all_doc_ids = set(tokenized_documents.keys())

def intersection(a, b):
    return sorted(list(set(a) & set(b)))


def union(a, b):
    return sorted(list(set(a) | set(b)))


def get_not(word, postings_index, all_doc_ids):
    posting_list = get_posting(word, postings_index)
    return sorted(list(all_doc_ids.difference(set(posting_list))))

## Parsing molto semplice della query

Costruiamo ora una prima funzione che separa:

- gli operatori booleani
- i termini della query

Questa implementazione è volutamente minimale:
- riconosce `and`, `or`, `not`
- preprocessa i termini
- non gestisce ancora parentesi
- non gestisce una vera precedenza tra operatori

In [ ]:
def generate_command_tokens(query):
    query = query.lower()
    raw_tokens = word_tokenize(query)

    commands = []
    query_words = []

    for token in raw_tokens:
        if token in ["and", "or", "not"]:
            commands.append(token)
        else:
            processed_tokens = preprocess(token, is_query=True)
            if len(processed_tokens) > 0:
                query_words.append(processed_tokens[0])

    return commands, query_words

## Gestione dell'operatore NOT

L’operatore `NOT` è leggermente più delicato degli altri, perché non combina due posting lists già esistenti, ma costruisce il complemento di una posting list rispetto all’insieme di tutti i documenti.

Nel codice seguente usiamo una strategia semplice:
- identifichiamo i `NOT`
- calcoliamo il complemento del termine successivo
- memorizziamo temporaneamente il risultato

In [ ]:
def gen_not_tuple(query_words, commands, postings_index, all_doc_ids):
    not_result = set()

    while "not" in commands:
        i = commands.index("not")
        word = query_words[i]

        complemented_docs = get_not(word, postings_index, all_doc_ids)
        not_result.update(complemented_docs)

        commands.pop(i)
        query_words[i] = i  # segnaposto temporaneo

        #print("\nAfter NOT processing:", commands, query_words)

    return not_result

## Combinazione delle posting lists

Una volta preprocessata la query e gestiti gli eventuali `NOT`, possiamo applicare gli operatori binari `AND` e `OR`.

Questa implementazione segue l’ordine con cui gli operatori compaiono nella query e non applica ancora nessuna strategia avanzata di parsing.

In [ ]:
def binary_operations(query_words, commands, not_docs, postings_index):
    if len(query_words) == 0:
        return []

    first_term = query_words[0]
    if isinstance(first_term, int):
        current_result = sorted(list(not_docs))
    else:
        current_result = get_posting(first_term, postings_index)

    remaining_query_words = query_words[1:]

    for i, command in enumerate(commands):
        next_item = remaining_query_words[i]

        if isinstance(next_item, int):
            next_result = sorted(list(not_docs))
        else:
            next_result = get_posting(next_item, postings_index)

        if command == "and":
            current_result = intersection(current_result, next_result)
        elif command == "or":
            current_result = union(current_result, next_result)
        else:
            print("Invalid command:", command)

    return current_result

## Funzione completa di esecuzione della query

Mettiamo ora insieme tutti i passaggi:
- parsing della query
- gestione del `NOT`
- applicazione di `AND` e `OR`
- stampa del risultato finale

In [ ]:
def execute_query(query, postings_index, all_doc_ids):
    print("\nInputQuery:", query)

    commands, query_words = generate_command_tokens(query)
    print("\nBEFORE invoking gen_not_tuple")
    print("Commands:", commands)
    print("Query Words:", query_words)

    not_docs = gen_not_tuple(query_words, commands, postings_index, all_doc_ids)

    print("\nAFTER invoking gen_not_tuple")
    print("Commands:", commands)
    print("Query Words:", query_words)
    print("NOT set size:", len(not_docs))

    final_result = binary_operations(query_words, commands, not_docs, postings_index)
    final_result = sorted(final_result)

    print("\nFinal Set:", final_result)

    print("Number of Retrieved Docs:", len(final_result))
    print("-" * 50)

    return final_result

In [ ]:
execute_query("place", postings, all_doc_ids)

execute_query("authority", postings, all_doc_ids)

execute_query("place and authority", postings, all_doc_ids)

execute_query("place or authority", postings, all_doc_ids)

execute_query("place and not authority", postings, all_doc_ids)

execute_query("place or not authority", postings, all_doc_ids)

print("\nNumber of documents in the original collection", len(all_doc_ids))


InputQuery: place

BEFORE invoking gen_not_tuple
Commands: []
Query Words: ['place']

AFTER invoking gen_not_tuple
Commands: []
Query Words: ['place']
NOT set size: 0

Final Set: [31, 39, 53, 59, 61, 76, 128, 192, 197, 200, 213, 228, 230, 231, 269, 274, 344, 345, 349, 366, 369, 370, 378, 438, 476, 499, 505, 513, 552, 561, 564]
Number of Retrieved Docs: 31
--------------------------------------------------

InputQuery: authority

BEFORE invoking gen_not_tuple
Commands: []
Query Words: ['author']

AFTER invoking gen_not_tuple
Commands: []
Query Words: ['author']
NOT set size: 0

Final Set: [11, 34, 44, 61, 69, 71, 76, 103, 142, 166, 167, 169, 175, 198, 209, 213, 234, 295, 351, 366, 381, 425, 502, 517, 550, 575]
Number of Retrieved Docs: 26
--------------------------------------------------

InputQuery: place and authority

BEFORE invoking gen_not_tuple
Commands: ['and']
Query Words: ['place', 'author']

AFTER invoking gen_not_tuple
Commands: ['and']
Query Words: ['place', 'author']
NOT 

## Query più complesse

Proviamo ora alcune query più complesse.

Questa parte è molto utile perché mostra sia il comportamento del sistema, sia alcuni limiti della nostra implementazione.

In [ ]:
execute_query("place or authority and welcome", postings, all_doc_ids)

execute_query("place or authority and not welcome", postings, all_doc_ids)

print("")


InputQuery: place or authority and welcome

BEFORE invoking gen_not_tuple
Commands: ['or', 'and']
Query Words: ['place', 'author', 'welcom']

AFTER invoking gen_not_tuple
Commands: ['or', 'and']
Query Words: ['place', 'author', 'welcom']
NOT set size: 0

Final Set: [11, 366, 499, 575]
Number of Retrieved Docs: 4
--------------------------------------------------

InputQuery: place or authority and not welcome

BEFORE invoking gen_not_tuple
Commands: ['or', 'and', 'not']
Query Words: ['place', 'author', 'welcom']

AFTER invoking gen_not_tuple
Commands: ['or', 'and']
Query Words: ['place', 'author', 2]
NOT set size: 574

Final Set: [31, 34, 39, 44, 53, 59, 61, 69, 71, 76, 103, 128, 142, 166, 167, 169, 175, 192, 197, 198, 200, 209, 213, 228, 230, 231, 234, 269, 274, 295, 344, 345, 349, 351, 369, 370, 378, 381, 425, 438, 476, 502, 505, 513, 517, 550, 552, 561, 564]
Number of Retrieved Docs: 49
--------------------------------------------------



## Interpretazione dei risultati

La funzione produce un insieme di `docID` che soddisfano la query booleana secondo l’implementazione attuale.

A questo punto può essere utile leggere il contenuto di alcuni documenti restituiti, per verificare se i risultati sono plausibili.

In [ ]:
def print_document(doc_id, documents, max_chars=3000):
    print(f"Document {doc_id}\n")
    print(documents[doc_id][:max_chars])

In [ ]:
results = execute_query("place and authority", postings, all_doc_ids)

if len(results) > 0:
    print_document(results[0], documents, max_chars=100000)


InputQuery: place and authority

Commands: ['and']

Query Words: ['place', 'author']

NOT set size: 0

Final Set: [61, 76, 213, 366]
--------------------------------------------------
Document 61

From: jk87377@lehtori.cc.tut.fi (Kouhia Juhana)
Subject: XV problems
Organization: Tampere University of Technology
Lines: 113
Distribution: world
NNTP-Posting-Host: cc.tut.fi

[Please, note the Newsgroups.]

Recent discussion about XV's problems were held in some newsgroup.
Here is some text users of XV might find interesting.
I have added more to text to this collection article, so read on, even
you so my articles a while ago.

I hope author of XV corrects those problems as best he can, so fine
program XV is that it is worth of improving.
(I have also minor ideas for 24bit XV, e-mail me for them.)

Any misundertanding of mine is understandable.


Juhana Kouhia


==clip==

[ ..deleted..]

Note that 'xv' saves only 8bit/rasterized images; that means that
the saved jpegs are just like jpeg-to

## Limite importante: precedenza degli operatori

Consideriamo la query:

`place or authority and welcome`

Nel nostro sistema attuale, gli operatori vengono applicati nell’ordine in cui compaiono.

Questo significa che la query **non** viene interpretata con una vera precedenza logica tra `AND` e `OR`.

### Conseguenza
Il risultato può essere diverso da quello che ci aspetteremmo in un parser booleano più rigoroso.

Questo è un limite voluto della nostra implementazione: il codice è semplice, ma la semantica delle query non è ancora del tutto corretta.

In [ ]:
query = "place or authority and welcome"
result = execute_query(query, postings, all_doc_ids)

print("\nNumber of retrieved documents:", len(result))


InputQuery: place or authority and welcome

Commands: ['or', 'and']

Query Words: ['place', 'author', 'welcom']

NOT set size: 0

Final Set: [11, 366, 499, 575]
--------------------------------------------------

Number of retrieved documents: 4


## Discussione tecnica

La nostra implementazione funziona, ma presenta diversi limiti interessanti dal punto di vista dell’Information Retrieval e del software engineering.

### Alcuni limiti evidenti
- usa operazioni su `set` invece di sfruttare postings lists ordinate con merge
- non gestisce parentesi
- non applica una vera precedenza tra operatori
- gestisce `NOT` in modo semplice ma non particolarmente elegante
- non valida la sintassi della query

Questi limiti non sono un difetto del laboratorio: sono anzi una parte importante dell’esperienza didattica, perché mostrano il divario tra una prima implementazione funzionante e un sistema più robusto.

## Mini esercizio

Prova a eseguire altre query, ad esempio:

- `graphic and file`
- `image or format`
- `window and not file`
- `graphic or image and color`

Per ciascuna query chiediti:

1. il risultato sembra plausibile?
2. la forma preprocessata dei termini è quella attesa?
3. la mancanza di parentesi o precedenza può alterare il risultato?

## Verso una versione migliore

Nel prossimo passo potremmo migliorare il sistema in almeno quattro direzioni:

1. usare postings lists ordinate e intersezione tramite **merge**
2. implementare correttamente la precedenza tra `AND`, `OR` e `NOT`
3. supportare le parentesi nelle query
4. validare la sintassi della query prima dell’esecuzione

Questi miglioramenti sono lasciati come esercizi o possibili estensioni del laboratorio.

## Esercizi finali

In questa parte del laboratorio proponiamo alcune estensioni utili per trasformare il prototipo costruito fin qui in un sistema più corretto ed efficiente.

Gli esercizi possono essere affrontati in ordine crescente di difficoltà.

### Esercizio 1 — Ottimizzare l'operatore NOT

L'implementazione attuale di `NOT` costruisce il complemento di una posting list rispetto all'insieme di tutti i documenti usando operazioni su insiemi.

#### Obiettivo
Prova a migliorare questa parte del codice, discutendo:

- il costo computazionale della soluzione attuale
- possibili alternative
- il ruolo del numero totale di documenti nella complessità dell'operazione

#### Domanda guida
È sempre necessario costruire esplicitamente l'insieme complemento?

### Esercizio 2 — Introdurre la precedenza degli operatori

Nel sistema attuale, la query viene valutata da sinistra a destra.

Questo significa che una query come:

`place or authority and welcome`

non viene interpretata secondo le regole standard della logica booleana.

#### Obiettivo
Modifica il sistema in modo che:
- `NOT` abbia precedenza più alta
- `AND` abbia precedenza più alta di `OR`

#### Suggerimento
Puoi affrontare il problema in diversi modi:
- riscrivendo il parser
- trasformando la query in una rappresentazione intermedia
- oppure applicando gli operatori in più passaggi

### Esercizio 3 — Supportare le parentesi

Un sistema booleano più realistico dovrebbe essere in grado di gestire query come:

- `(place or authority) and welcome`
- `place and (not authority)`
- `(graphic or image) and (file or format)`

#### Obiettivo
Estendi il parser in modo da riconoscere ed elaborare correttamente le parentesi.

#### Nota
Questo esercizio richiede di distinguere in modo più rigoroso tra:
- token lessicali
- operatori booleani
- struttura sintattica della query

### Esercizio 4 — Validare la sintassi delle query

Attualmente il sistema assume che la query in input sia ben formata.

Tuttavia, un utente potrebbe scrivere query problematiche, ad esempio:

- `and place authority`
- `place or or authority`
- `not`
- `( place and authority`

#### Obiettivo
Aggiungi una fase di validazione sintattica che intercetti query non valide prima dell'esecuzione.

#### Domanda guida
Quali sono i casi di errore più semplici da rilevare?

### Esercizio 5 — Sostituire i set con il merge di posting lists

Nel laboratorio precedente abbiamo visto come intersecare due posting lists ordinate tramite una procedura di **merge**.

#### Obiettivo
Riscrivi gli operatori `AND` e `OR` usando direttamente postings lists ordinate, evitando di trasformarle ogni volta in `set`.

#### Perché è interessante?
Questo esercizio avvicina il sistema a un'implementazione più fedele ai principi classici dell'Information Retrieval.

### Esercizio 6 — Analisi qualitativa del preprocessing

Il preprocessing scelto in questo notebook include:
- rimozione dell'header
- lowercase
- stopword removal
- stemming
- rimozione della punteggiatura
- normalizzazione dei numeri

#### Obiettivo
Scegli alcuni documenti e osserva come cambia la loro rappresentazione dopo il preprocessing.

#### Domande guida
- quali trasformazioni sembrano utili?
- quali potrebbero eliminare informazione rilevante?
- in quali casi lo stemming potrebbe essere troppo aggressivo?

### Esercizio 7 — Analisi della selettività dei termini

Usa la document frequency per confrontare termini diversi della collezione.

#### Obiettivo
Identifica:
- termini molto frequenti
- termini molto rari
- termini probabilmente più utili per query selettive

#### Domanda guida
In che modo la document frequency può aiutare a ottimizzare il query processing?

## Possibili estensioni del laboratorio

Il sistema costruito in questo notebook è volutamente semplice, ma può essere esteso in molti modi.

### Alcune direzioni possibili
- aggiungere il supporto a **phrase queries**
- costruire un **positional index**
- introdurre un parser booleano completo
- usare merge ottimizzati su postings ordinate
- confrontare diverse pipeline di preprocessing
- passare da retrieval booleano a retrieval con **ranking**

Queste estensioni mostrano come si possa passare gradualmente da un prototipo didattico a un sistema più vicino a un vero motore di ricerca.

## Riassunto del laboratorio

In questo laboratorio abbiamo affrontato una versione più realistica del problema dell'Information Retrieval, lavorando su una collezione vera di documenti testuali.

### Abbiamo visto come
- caricare una collezione reale
- analizzare documenti grezzi
- progettare una pipeline di preprocessing
- costruire un **indice inverso non posizionale**
- salvare e ricaricare l'indice
- eseguire query booleane semplici e complesse

### Abbiamo anche osservato che
una prima implementazione funzionante non coincide necessariamente con una implementazione ottimale o formalmente completa.

Questo è un punto didatticamente importante: costruire un prototipo semplice permette di capire meglio i problemi che emergono quando si progettano sistemi di retrieval reali.

## Conclusione

Con questo laboratorio abbiamo fatto un passo importante dal modello concettuale visto nel primo notebook a una implementazione concreta su una collezione reale.

Nel seguito del corso, questi stessi concetti potranno essere estesi in diverse direzioni:
- indici posizionali
- query di frase e di prossimità
- ottimizzazione del query processing
- modelli di ranking
- valutazione dell'efficacia del retrieval

In altre parole, abbiamo costruito una prima base su cui innestare versioni progressivamente più potenti dei sistemi di Information Retrieval.

## Take-home message

In questi primi due laboratori abbiamo visto che i concetti fondamentali dell’Information Retrieval possono essere compresi bene solo collegando **modello teorico** e **implementazione concreta**.

In particolare:

- una collezione testuale deve essere trasformata in una rappresentazione adatta alla ricerca
- l’**inverted index** è la struttura dati centrale che rende efficiente il retrieval booleano
- il **preprocessing** influenza in modo sostanziale ciò che il sistema sarà in grado di trovare
- l’esecuzione di una query dipende non solo dai dati, ma anche dalle scelte progettuali sul linguaggio di query e sugli algoritmi di processing

Il messaggio principale è quindi questo:  
un sistema di Information Retrieval non è solo un insieme di documenti e parole, ma il risultato di una serie di decisioni su **rappresentazione**, **indicizzazione** e **interpretazione delle query**.

Capire queste scelte è il primo passo per progettare sistemi di ricerca più efficaci, più robusti e più vicini ai motori di ricerca reali.